Cell 1 — Imports

In [42]:
import numpy as np
import pandas as pd
from scipy.stats import truncnorm

Cell 2 — Reproducibility and dataset size

In [43]:
SEED = 42
np.random.seed(SEED)

N_PER_CLASS = 5000
CLASS_NAMES = ["Healthy", "Asthma", "Pneumonia"]

Cell 3 — Helper functions

In [44]:
def truncnorm_sample(mean, sd, low, high, size=1):
    a = (low - mean) / sd
    b = (high - mean) / sd
    return truncnorm.rvs(a, b, loc=mean, scale=sd, size=size)

def weighted_choice(prob_dict):
    vals = list(prob_dict.keys())
    probs = list(prob_dict.values())
    return np.random.choice(vals, p=probs)

def zero_inflated_poisson(size, p_zero=0.35, lam=1.4, max_val=7):
    zeros = np.random.binomial(1, p_zero, size=size)
    vals = np.where(zeros == 1, 0, np.random.poisson(lam, size=size))
    return np.clip(vals, 0, max_val)

def shifted_neg_binomial(size, mean_target, dispersion, min_val=1, max_val=35):
    p = dispersion / (dispersion + mean_target)
    vals = np.random.negative_binomial(dispersion, p, size=size) + min_val
    return np.clip(vals, min_val, max_val)

def resample_more_severe():
    return np.random.choice([1, 2, 3], p=[0.10, 0.40, 0.50])

Cell 4 — Revised v5 rule table

In [45]:
rules = {
    "Healthy": {
        "age": (38, 12, 18, 65),
        "sex_probs": {"Female": 0.50, "Male": 0.50},
        "smoking_probs": {0: 0.65, 1: 0.20, 2: 0.15},
        "heart_rate": (77, 9, 58, 102),
        "respiratory_rate": (16.0, 2.5, 11, 24),
        "temperature": (36.9, 0.30, 36.0, 37.8),
        "spo2": (97.4, 1.4, 93, 100),
        "cough_probs": {0: 0.48, 1: 0.34, 2: 0.14, 3: 0.04},
        "dyspnea_probs": {0: 0.62, 1: 0.25, 2: 0.10, 3: 0.03},
        "fev1_fvc": (81.5, 5.5, 70, 95),
        "severity_probs": {0: 0.55, 1: 0.30, 2: 0.15}
    },

    "Asthma": {
        "age": (35, 13, 18, 60),
        "sex_probs": {"Female": 0.60, "Male": 0.40},
        "smoking_probs": {0: 0.55, 1: 0.30, 2: 0.15},
        "heart_rate": (87, 11, 60, 124),
        "respiratory_rate": (20.5, 4.5, 12, 34),
        "temperature": (37.0, 0.32, 36.0, 38.0),
        "spo2": (95.0, 2.2, 88, 100),
        "cough_probs": {0: 0.10, 1: 0.28, 2: 0.38, 3: 0.24},
        "dyspnea_probs": {0: 0.10, 1: 0.28, 2: 0.37, 3: 0.25},
        "fev1_fvc": (71.5, 9.0, 45, 88),
        "severity_probs": {0: 0.25, 1: 0.30, 2: 0.30, 3: 0.15}
    },

    "Pneumonia": {
        "age": (52, 17, 18, 85),
        "sex_probs": {"Female": 0.45, "Male": 0.55},
        "smoking_probs": {0: 0.40, 1: 0.30, 2: 0.30},
        "heart_rate": (95, 12, 68, 132),
        "respiratory_rate": (22.5, 4.5, 14, 36),
        "temperature": (37.8, 0.65, 36.6, 40.2),
        "spo2": (93.0, 2.6, 84, 99),
        "cough_probs": {0: 0.04, 1: 0.18, 2: 0.42, 3: 0.36},
        "dyspnea_probs": {0: 0.06, 1: 0.20, 2: 0.40, 3: 0.34},
        "fev1_fvc": (76.5, 6.5, 60, 90),
        "severity_probs": {0: 0.35, 1: 0.45, 2: 0.20}
    }
}

Cell 5 — Revised row generator for v5

In [46]:
def generate_row(target_class):
    r = rules[target_class]
    severity = weighted_choice(r["severity_probs"])

    age = int(round(truncnorm_sample(*r["age"], size=1)[0]))
    sex = weighted_choice(r["sex_probs"])
    smoking_status = int(weighted_choice(r["smoking_probs"]))

    heart_rate = truncnorm_sample(*r["heart_rate"], size=1)[0]
    respiratory_rate = truncnorm_sample(*r["respiratory_rate"], size=1)[0]
    temperature = truncnorm_sample(*r["temperature"], size=1)[0]
    spo2 = truncnorm_sample(*r["spo2"], size=1)[0]
    cough = int(weighted_choice(r["cough_probs"]))
    dyspnea = int(weighted_choice(r["dyspnea_probs"]))
    fev1_fvc = truncnorm_sample(*r["fev1_fvc"], size=1)[0]

    if target_class == "Healthy":
        symptom_duration = int(zero_inflated_poisson(1, p_zero=0.35, lam=1.5, max_val=7)[0])

    elif target_class == "Asthma":
        symptom_duration = int(shifted_neg_binomial(1, mean_target=7, dispersion=4, min_val=1, max_val=28)[0])

    else:
        symptom_duration = int(shifted_neg_binomial(1, mean_target=5, dispersion=4, min_val=1, max_val=18)[0])

    # Hidden state meanings
    # Healthy: 0 = clearly normal, 1 = mild nonspecific respiratory noise, 2 = overlap-prone healthy
    # Asthma: 0 = between episodes, 1 = mild flare, 2 = moderate flare, 3 = severe exacerbation
    # Pneumonia: 0 = mild/early atypical CAP, 1 = typical CAP, 2 = severe CAP

    if target_class == "Healthy":
        if severity == 0:
            cough = min(cough, np.random.choice([0, 1], p=[0.65, 0.35]))
            dyspnea = min(dyspnea, np.random.choice([0, 1], p=[0.80, 0.20]))
            symptom_duration = min(symptom_duration, np.random.choice([0, 1, 2], p=[0.45, 0.35, 0.20]))
            spo2 += np.random.uniform(0, 0.8)

        elif severity == 1:
            cough = max(cough, np.random.choice([0, 1, 2], p=[0.20, 0.55, 0.25]))
            dyspnea = max(dyspnea, np.random.choice([0, 1, 2], p=[0.40, 0.45, 0.15]))
            respiratory_rate += np.random.uniform(0.5, 2.0)
            heart_rate += np.random.uniform(0, 4)
            temperature += np.random.uniform(0.0, 0.25)
            symptom_duration = max(symptom_duration, np.random.choice([1, 2, 3, 4], p=[0.25, 0.35, 0.25, 0.15]))

        elif severity == 2:
            cough = max(cough, np.random.choice([1, 2, 3], p=[0.45, 0.40, 0.15]))
            dyspnea = max(dyspnea, np.random.choice([1, 2, 3], p=[0.50, 0.35, 0.15]))
            respiratory_rate += np.random.uniform(1.0, 3.0)
            heart_rate += np.random.uniform(1.0, 5.0)
            temperature += np.random.uniform(0.1, 0.45)
            spo2 -= np.random.uniform(0.0, 1.5)
            symptom_duration = max(symptom_duration, np.random.choice([2, 3, 4, 5], p=[0.25, 0.35, 0.25, 0.15]))

    elif target_class == "Asthma":
        if severity == 0:
            cough = min(max(cough, 1), np.random.choice([1, 2], p=[0.70, 0.30]))
            dyspnea = min(max(dyspnea, 0), np.random.choice([0, 1, 2], p=[0.30, 0.50, 0.20]))
            heart_rate += np.random.uniform(-2, 2)
            respiratory_rate += np.random.uniform(-1, 1.5)
            spo2 += np.random.uniform(0.0, 1.5)
            fev1_fvc += np.random.uniform(3, 8)
            temperature += np.random.uniform(-0.1, 0.15)

        elif severity == 1:
            respiratory_rate += np.random.uniform(1, 3)
            heart_rate += np.random.uniform(2, 5)
            spo2 -= np.random.uniform(0.3, 1.2)
            fev1_fvc -= np.random.uniform(1, 4)
            cough = max(cough, np.random.choice([1, 2, 3], p=[0.18, 0.56, 0.26]))
            dyspnea = max(dyspnea, np.random.choice([1, 2, 3], p=[0.20, 0.55, 0.25]))

        elif severity == 2:
            respiratory_rate += np.random.uniform(2, 5)
            heart_rate += np.random.uniform(4, 8)
            spo2 -= np.random.uniform(1.0, 2.5)
            fev1_fvc -= np.random.uniform(4, 8)
            cough = max(cough, np.random.choice([1, 2, 3], p=[0.08, 0.47, 0.45]))
            dyspnea = max(dyspnea, np.random.choice([1, 2, 3], p=[0.08, 0.42, 0.50]))

        elif severity == 3:
            respiratory_rate += np.random.uniform(4, 7)
            heart_rate += np.random.uniform(6, 11)
            spo2 -= np.random.uniform(2.0, 4.0)
            fev1_fvc -= np.random.uniform(7, 12)
            cough = max(cough, np.random.choice([1, 2, 3], p=[0.05, 0.35, 0.60]))
            dyspnea = max(dyspnea, np.random.choice([1, 2, 3], p=[0.05, 0.30, 0.65]))

    elif target_class == "Pneumonia":
        if severity == 0:
            temperature += np.random.uniform(-0.2, 0.25)
            respiratory_rate += np.random.uniform(0.5, 2.5)
            heart_rate += np.random.uniform(0, 4)
            spo2 -= np.random.uniform(0.2, 1.5)
            cough = max(cough, np.random.choice([1, 2, 3], p=[0.20, 0.55, 0.25]))
            dyspnea = max(dyspnea, np.random.choice([1, 2, 3], p=[0.25, 0.55, 0.20]))

        elif severity == 1:
            temperature += np.random.uniform(0.10, 0.50)
            respiratory_rate += np.random.uniform(2, 4.5)
            heart_rate += np.random.uniform(2, 6)
            spo2 -= np.random.uniform(1.0, 2.2)
            cough = max(cough, np.random.choice([1, 2, 3], p=[0.10, 0.45, 0.45]))
            dyspnea = max(dyspnea, np.random.choice([1, 2, 3], p=[0.10, 0.42, 0.48]))

        elif severity == 2:
            temperature += np.random.uniform(0.35, 0.90)
            respiratory_rate += np.random.uniform(4, 7)
            heart_rate += np.random.uniform(5, 10)
            spo2 -= np.random.uniform(2.0, 4.0)
            cough = max(cough, np.random.choice([1, 2, 3], p=[0.06, 0.34, 0.60]))
            dyspnea = max(dyspnea, np.random.choice([1, 2, 3], p=[0.05, 0.30, 0.65]))

    # Dependency rules
    if target_class == "Pneumonia" and temperature > 38.5 and np.random.rand() < 0.70:
        heart_rate += np.random.uniform(3, 8)

    if target_class == "Pneumonia" and spo2 < 92:
        dyspnea = max(dyspnea, resample_more_severe())

    if target_class == "Asthma" and severity == 3:
        fev1_fvc -= np.random.uniform(0, 2)
        respiratory_rate += np.random.uniform(0, 1.5)

    if target_class == "Asthma" and severity == 0 and np.random.rand() < 0.65:
        fev1_fvc += np.random.uniform(2, 5)
        spo2 += np.random.uniform(0, 1.0)

    if spo2 < 90:
        dyspnea = max(dyspnea, 1)

    if symptom_duration == 0:
        cough = min(cough, np.random.choice([0, 1], p=[0.65, 0.35]))
        dyspnea = min(dyspnea, np.random.choice([0, 1], p=[0.80, 0.20]))

    if target_class == "Healthy":
        if temperature > 37.6 and spo2 < 94 and cough >= 2:
            spo2 = max(94, spo2)
            cough = min(cough, 2)
            dyspnea = min(dyspnea, 2)

    # Stronger overlap rules
    if target_class == "Pneumonia" and np.random.rand() < 0.25:
        temperature -= np.random.uniform(0.3, 1.0)
        respiratory_rate -= np.random.uniform(0, 2)
        spo2 += np.random.uniform(0, 1.5)

    if target_class == "Asthma" and np.random.rand() < 0.22:
        temperature += np.random.uniform(0.0, 0.4)
        symptom_duration = max(1, symptom_duration - np.random.randint(1, 5))
        fev1_fvc += np.random.uniform(1.5, 5.0)
        spo2 += np.random.uniform(0, 1.0)

    if target_class == "Healthy" and np.random.rand() < 0.12:
        cough = min(3, cough + np.random.choice([0, 1], p=[0.30, 0.70]))
        dyspnea = min(2, dyspnea + np.random.choice([0, 1], p=[0.40, 0.60]))
        respiratory_rate += np.random.uniform(0, 2)
        symptom_duration = max(symptom_duration, np.random.choice([1, 2, 3], p=[0.35, 0.40, 0.25]))

    # Smoking effects
    if smoking_status == 2:
        heart_rate += np.random.uniform(0, 2)
        if target_class in ["Asthma", "Pneumonia"]:
            cough = min(3, cough + np.random.choice([0, 1], p=[0.45, 0.55]))

    # Age effect in pneumonia
    if age >= 65 and target_class == "Pneumonia":
        spo2 -= np.random.uniform(0, 1.2)
        respiratory_rate += np.random.uniform(0, 2)

    # Final clipping
    if target_class == "Healthy":
        heart_rate = np.clip(heart_rate, 58, 104)
        respiratory_rate = np.clip(respiratory_rate, 11, 24)
        temperature = np.clip(temperature, 36.0, 37.8)
        spo2 = np.clip(spo2, 93, 100)
        fev1_fvc = np.clip(fev1_fvc, 70, 95)
        symptom_duration = np.clip(symptom_duration, 0, 7)

    elif target_class == "Asthma":
        heart_rate = np.clip(heart_rate, 60, 124)
        respiratory_rate = np.clip(respiratory_rate, 12, 34)
        temperature = np.clip(temperature, 36.0, 38.0)
        spo2 = np.clip(spo2, 88, 100)
        fev1_fvc = np.clip(fev1_fvc, 45, 88)
        symptom_duration = np.clip(symptom_duration, 1, 28)

    elif target_class == "Pneumonia":
        heart_rate = np.clip(heart_rate, 68, 132)
        respiratory_rate = np.clip(respiratory_rate, 14, 36)
        temperature = np.clip(temperature, 36.6, 40.2)
        spo2 = np.clip(spo2, 84, 99)
        fev1_fvc = np.clip(fev1_fvc, 60, 90)
        symptom_duration = np.clip(symptom_duration, 1, 18)

    return {
        "age": int(age),
        "sex": sex,
        "smoking_status": int(smoking_status),
        "heart_rate": int(round(heart_rate)),
        "respiratory_rate": int(round(respiratory_rate)),
        "temperature": round(float(temperature), 1),
        "spo2": int(round(spo2)),
        "cough_severity": int(cough),
        "dyspnea_severity": int(dyspnea),
        "fev1_fvc_ratio": round(float(fev1_fvc), 1),
        "symptom_duration_days": int(symptom_duration),
        "hidden_severity_state": int(severity),
        "target_class": target_class
    }

Cell 6 — Generate the 15,000-row v5 dataset

In [47]:
rows = []

for cls in CLASS_NAMES:
    for _ in range(N_PER_CLASS):
        rows.append(generate_row(cls))

df = pd.DataFrame(rows)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(df.shape)
df.head()

(15000, 13)


,age,sex,smoking_status,heart_rate,respiratory_rate,temperature,spo2,cough_severity,dyspnea_severity,fev1_fvc_ratio,symptom_duration_days,hidden_severity_state,target_class
0,80,Male,1,117,31,38.6,94,3,3,82.4,7,2,Pneumonia
1,42,Female,0,96,27,37.3,95,3,3,64.3,11,1,Asthma
2,49,Male,2,106,23,38.3,90,3,2,80.5,4,2,Pneumonia
3,32,Male,0,68,18,36.6,97,1,1,80.7,3,0,Healthy
4,44,Female,0,92,19,36.7,96,2,0,79.9,8,0,Asthma


Cell 7 — Add label columns

In [48]:
smoking_map = {
    0: "Never",
    1: "Ex-smoker",
    2: "Current smoker"
}

def severity_label(row):
    if row["target_class"] == "Healthy":
        return {
            0: "Clearly normal",
            1: "Mild nonspecific respiratory noise",
            2: "Overlap-prone healthy"
        }[row["hidden_severity_state"]]

    elif row["target_class"] == "Asthma":
        return {
            0: "Between episodes",
            1: "Mild flare",
            2: "Moderate flare",
            3: "Severe exacerbation"
        }[row["hidden_severity_state"]]

    else:
        return {
            0: "Mild/early atypical CAP",
            1: "Typical CAP",
            2: "Severe CAP"
        }[row["hidden_severity_state"]]

df["smoking_status_label"] = df["smoking_status"].map(smoking_map)
df["hidden_severity_label"] = df.apply(severity_label, axis=1)

df.head()

,age,sex,smoking_status,heart_rate,respiratory_rate,temperature,spo2,cough_severity,dyspnea_severity,fev1_fvc_ratio,symptom_duration_days,hidden_severity_state,target_class,smoking_status_label,hidden_severity_label
0,80,Male,1,117,31,38.6,94,3,3,82.4,7,2,Pneumonia,Ex-smoker,Severe CAP
1,42,Female,0,96,27,37.3,95,3,3,64.3,11,1,Asthma,Never,Mild flare
2,49,Male,2,106,23,38.3,90,3,2,80.5,4,2,Pneumonia,Current smoker,Severe CAP
3,32,Male,0,68,18,36.6,97,1,1,80.7,3,0,Healthy,Never,Clearly normal
4,44,Female,0,92,19,36.7,96,2,0,79.9,8,0,Asthma,Never,Between episodes


Cell 8 — Save the hidden-severity version

In [49]:
df.to_csv("synthetic_respiratory_dataset_15000_with_hidden_severity_v5.csv", index=False)
print("Saved: synthetic_respiratory_dataset_15000_with_hidden_severity_v5.csv")

Saved: synthetic_respiratory_dataset_15000_with_hidden_severity_v5.csv


Cell 9 — Export final modelling dataset

In [50]:
final_df = df.drop(columns=["hidden_severity_state", "hidden_severity_label"]).copy()
final_df.to_csv("synthetic_respiratory_dataset_15000_v51.csv", index=False)

print("Saved: synthetic_respiratory_dataset_15000_v5.csv")
final_df.head()

Saved: synthetic_respiratory_dataset_15000_v5.csv


,age,sex,smoking_status,heart_rate,respiratory_rate,temperature,spo2,cough_severity,dyspnea_severity,fev1_fvc_ratio,symptom_duration_days,target_class,smoking_status_label
0,80,Male,1,117,31,38.6,94,3,3,82.4,7,Pneumonia,Ex-smoker
1,42,Female,0,96,27,37.3,95,3,3,64.3,11,Asthma,Never
2,49,Male,2,106,23,38.3,90,3,2,80.5,4,Pneumonia,Current smoker
3,32,Male,0,68,18,36.6,97,1,1,80.7,3,Healthy,Never
4,44,Female,0,92,19,36.7,96,2,0,79.9,8,Asthma,Never


Cell 10 — Save summary statistics

In [51]:
summary_df = final_df.groupby("target_class")[[
    "age", "heart_rate", "respiratory_rate", "temperature",
    "spo2", "fev1_fvc_ratio", "symptom_duration_days"
]].agg(["mean", "std", "min", "max"])

summary_df.to_csv("synthetic_respiratory_dataset_15000_summary_v5.csv")
print("Saved: synthetic_respiratory_dataset_15000_summary_v5.csv")
summary_df

Saved: synthetic_respiratory_dataset_15000_summary_v5.csv


age                    heart_rate                      \
                 mean        std min max       mean        std min  max   
target_class                                                              
Asthma        36.6486   9.921452  18  60    91.8954  11.146556  60  124   
Healthy       38.7652  10.265844  18  65    78.4506   8.609529  58  104   
Pneumonia     51.5520  14.807767  18  85   100.4396  11.892276  71  132   

             respiratory_rate            ... spo2      fev1_fvc_ratio  \
                         mean       std  ...  min  max           mean   
target_class                             ...                            
Asthma                23.4146  4.565312  ...   88  100       68.96686   
Healthy               16.8752  2.485570  ...   93  100       81.69068   
Pneumonia             25.8058  4.507521  ...   84   99       76.32610   

                                    symptom_duration_days                    
                    std   min   max                  mean       std min max  
target_class                                                                 
Asthma        10.123821  45.0  88.0                7.4172  4.342547   1  28  
Healthy        5.026670  70.0  94.9                1.5536  1.473758   0   6  
Pneumonia      5.958448  60.0  90.0                6.0358  3.307069   1  18  

[3 rows x 28 columns]

Cell 11 — Quick categorical inspection

In [52]:
print(final_df["target_class"].value_counts())
print()
print(pd.crosstab(df["target_class"], df["hidden_severity_label"]))
print()
print(pd.crosstab(final_df["target_class"], final_df["smoking_status_label"]))

target_class
Pneumonia    5000
Asthma       5000
Healthy      5000
Name: count, dtype: int64

hidden_severity_label  Between episodes  Clearly normal  Mild flare  \
target_class                                                          
Asthma                             1192               0        1462   
Healthy                               0            2720           0   
Pneumonia                             0               0           0   

hidden_severity_label  Mild nonspecific respiratory noise  \
target_class                                                
Asthma                                                  0   
Healthy                                              1557   
Pneumonia                                               0   

hidden_severity_label  Mild/early atypical CAP  Moderate flare  \
target_class                                                     
Asthma                                       0            1602   
Healthy                                      0

Cell 12 — Final v5 file to use in the ML notebook

In [29]:
# This is the file to load in the ML notebook
print("Use this file in the next notebook:")
print("synthetic_respiratory_dataset_15000_v5.csv")

Use this file in the next notebook:
synthetic_respiratory_dataset_15000_v5.csv
